In [ ]:
import pyecap
import os
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.io as sio
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from flexNIRs.fnirs_functions import *

In [ ]:
meta_index = 2
metaDF = pd.read_excel(r'D:\Data\TCD\20260220_TCD_QC\meta.xlsx')
tank_folder = r'D:\Data\TCD\20260220_TCD_QC\CVP_SingleCh_Ramp-260220\\'

raw_ecg = pyecap.Ephys(tank_folder + metaDF.loc[meta_index, 'Tank'], stores = 'ECGG')
raw_ecg = raw_ecg.remove_ch(['ECGG 2','ECGG 3', 'ECGG 4'])

raw_tcd = pyecap.Ephys(tank_folder + metaDF.loc[meta_index, 'Tank'], stores = 'TCDD')
raw_tcd_fir = raw_tcd.filter_fir(cutoff = (1,10), width = 1, pass_zero = 'bandpass')

stim = pyecap.Stim(tank_folder + metaDF.loc[meta_index, 'Tank'])
stimDF = stim.parameters
#stimDF

In [ ]:
"""Generate arrays and Down-sample for plotting"""
DS_factor = 20

ecg_d = raw_ecg.array[0,:].compute()
tcd_d = signal.medfilt(raw_tcd.array[0,:].compute(), kernel_size = 3)  #Applies 3 window median filter to remove TCD data drops
tcd_fir = raw_tcd_fir.array[0,:].compute()
time = raw_ecg.time()

ecg_f = tdt_ecg = filter_hr(raw_ecg, cutoff = 100, downsample=False, check_plot=False)[0] *1e-6
tcd_f = filter_hr(raw_tcd, cutoff = 100, downsample=False, check_plot=False)[0] *1e-6

tcd_DS = signal.decimate(tcd_d, DS_factor, zero_phase = True)
tcd_fir_DS = signal.decimate(tcd_fir, DS_factor, zero_phase = True)
time_ds = np.arange(len(tcd_DS)) * (DS_factor / raw_tcd.sample_rate)

tcd_DS_f = filter_hr(raw_tcd, cutoff = 10, downsample=True, downsample_factor = DS_factor, check_plot=False)[0] *1e-6

In [ ]:
"""Perform Calculations on DS arrays"""
dvt_f = np.gradient(tcd_DS_f, DS_factor / raw_tcd.sample_rate) ** 3
dvt_r = np.gradient(tcd_DS, DS_factor / raw_tcd.sample_rate) ** 3

In [ ]:
"""Plots"""
fig = make_subplots(specs = [[{'secondary_y' : True}]])
fig.update_layout(spikedistance = 0)

fig.add_trace(go.Scattergl(x = time_ds, y = tcd_DS_f, name='Gauss Filtered DS'))
#fig.add_trace(go.Scattergl(x = time_ds, y = tcd_fir_DS, name='FIR Filtered DS'))
fig.add_trace(go.Scattergl(x = time, y = tcd_d, name='Raw DS'))
#fig.add_trace(go.Scattergl(x = time_ds, y = dvt_f, name='1st Dvt - Filtered Data'), secondary_y = True)
#fig.add_trace(go.Scattergl(x = time_ds, y = dvt_r, name='1st Dvt - Raw Data'), secondary_y = True)

fig.show()